# RSNA Pulmonary Embolism Detection — review notebook

This reproducible prototype notebook uses a saved study-level split manifest, validates DICOM identifiers, converts pixels to HU and windows them, and compares frozen ResNet18 features with partial fine-tuning. The completed run used sampled studies and capped negative slices. Patient-level separation could not be verified because inspected DICOM headers lacked usable patient identifiers.

The notebook source is updated for review and future reruns; its cleared outputs are not a claim that this source was rerun. Historical metrics and curves are supplied in the review package.

## 1. Install dependencies, mount Drive, and set paths

Run in Colab with a GPU runtime. The path discovery below handles the shared-folder shortcut location used for this project.

In [1]:

# ============================================================
# RSNA Pulmonary Embolism Detection
# Environment: Windows Server 2025 + NVIDIA Tesla T4
# Python: Micromamba / AIProgram
# Dataset: Google Drive mounted as R:
# ============================================================

import os, json, random, shutil, sys, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom
from PIL import Image
from tqdm.auto import tqdm
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score,
    precision_score, recall_score, f1_score, confusion_matrix
)

# ============================================================
# 1. Dataset Paths
# ============================================================

# ORIGINAL: Dataset metadata remains on Google Drive
DATA_ROOT = Path(r"R:\RSNA STR")

CSV_PATH = DATA_ROOT / "train.csv"

# MODIFIED: Read DICOM images from the local server instead
# of Google Drive.
TRAIN_DIR = Path(r"D:\RSNA_DICOM")

if not CSV_PATH.is_file():
    raise FileNotFoundError(
        f"CSV file not found: {CSV_PATH}"
    )

if not TRAIN_DIR.is_dir():
    raise FileNotFoundError(
        f"Training directory not found: {TRAIN_DIR}"
    )

print("Dataset paths verified successfully!")

# ============================================================
# 2. Project Configuration
# ============================================================

SEED = 42
MAX_STUDIES = 600
MAX_NEG_TRAIN_PER_STUDY = 24
MAX_NEG_EVAL_PER_STUDY = 12
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 2

# ORIGINAL: Kept unchanged as requested.
NUM_WORKERS = 2

WINDOW_CENTER, WINDOW_WIDTH = 100.0, 700.0

# ============================================================
# 3. Local Output and Cache Directories
# ============================================================

PROJECT_DIR = Path(r"C:\RSNA_Project")

OUTPUT_DIR = PROJECT_DIR / "results"

# MODIFIED: Cache stored on the server's temporary D: drive.
LOCAL_CACHE = Path(r"D:\RSNA_Cache")

(OUTPUT_DIR / "checkpoints").mkdir(
    parents=True, exist_ok=True
)
LOCAL_CACHE.mkdir(parents=True, exist_ok=True)

# ============================================================
# 4. Reproducibility
# ============================================================

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ============================================================
# 5. GPU Configuration
# ============================================================

DEVICE = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

torch.backends.cudnn.benchmark = DEVICE.type == 'cuda'
torch.backends.cuda.matmul.allow_tf32 = DEVICE.type == 'cuda'
torch.backends.cudnn.allow_tf32 = DEVICE.type == 'cuda'

# ============================================================
# 6. Environment Verification
# ============================================================

print({
    'device': str(DEVICE),
    'gpu': (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available() else None
    ),
    'csv': str(CSV_PATH),
    'train_dir': str(TRAIN_DIR),
    'output_dir': str(OUTPUT_DIR),
    'cache_dir': str(LOCAL_CACHE)
})

# ADDED: Verify one locally downloaded DICOM file.
local_example = next(TRAIN_DIR.rglob("*.dcm"), None)

assert local_example is not None, (
    "No local DICOM files found in D:\\RSNA_DICOM"
)

print("Local DICOM example:", local_example)


FileNotFoundError: CSV file not found: R:\RSNA STR\train.csv

## 2. Validate labels and replay the saved study sample

The saved manifest is used so the review notebook points to the same 420/90/90 study split as the completed run. DICOM paths are built from competition identifiers, not recursive file scans.

In [12]:

# ============================================================
# STEP 2: Validate Labels and Replay the Saved Study Sample
# ============================================================

# ------------------------------------------------------------
# 1. Load required columns from the RSNA training CSV
# ------------------------------------------------------------

required = [
    'StudyInstanceUID',
    'SeriesInstanceUID',
    'SOPInstanceUID',
    'pe_present_on_image',
    'negative_exam_for_pe'
]

labels = pd.read_csv(CSV_PATH, usecols=required)

keys = required[:3]


# ------------------------------------------------------------
# 2. Validate DICOM identifiers
# ------------------------------------------------------------

# Check that Study, Series and SOP identifiers are not missing
assert not labels[keys].isna().any().any(), \
    'Null DICOM identifiers found.'

# Check that the combination of identifiers is unique
assert not labels.duplicated(keys).any(), \
    'Duplicate study/series/SOP keys found.'


# ------------------------------------------------------------
# 3. Validate binary classification labels
# ------------------------------------------------------------

for col in required[3:]:

    # Convert labels into numeric values
    labels[col] = pd.to_numeric(
        labels[col],
        errors='coerce'
    )

    # Ensure that labels contain only 0 and 1
    assert (
        labels[col].notna().all()
        and set(labels[col].unique()) <= {0, 1}
    ), f'Invalid labels in {col}.'

    # Reduce memory usage
    labels[col] = labels[col].astype('int8')


# ------------------------------------------------------------
# 4. Create study-level target
# ------------------------------------------------------------

# negative_exam_for_pe = 1 -> study_target = 0
# negative_exam_for_pe = 0 -> study_target = 1

labels['study_target'] = 1 - labels['negative_exam_for_pe']

# Ensure all slices within a study have the same study label
assert (
    labels.groupby('StudyInstanceUID')['negative_exam_for_pe']
    .nunique() == 1
).all(), 'Conflicting official study flags.'


# ------------------------------------------------------------
# 5. Create one record per CT study
# ------------------------------------------------------------

studies = labels.groupby(
    'StudyInstanceUID',
    as_index=False
).agg(
    study_target=('study_target', 'first')
)


# ------------------------------------------------------------
# 6. Load the original saved split manifest
# ------------------------------------------------------------

manifest_path = OUTPUT_DIR / 'split_manifest.csv'

assert manifest_path.is_file(), \
    f'Missing saved split manifest: {manifest_path}'

manifest = pd.read_csv(
    manifest_path,
    dtype={
        'StudyInstanceUID': str,
        'split': str
    }
)


# ------------------------------------------------------------
# 7. Validate Train / Validation / Test split
# ------------------------------------------------------------

# Required columns must exist
assert set(manifest.columns) >= {
    'StudyInstanceUID',
    'split'
}

# Each study should appear only once
assert not manifest.StudyInstanceUID.duplicated().any()

# Ensure all three dataset partitions exist
assert set(manifest.split) == {
    'train',
    'validation',
    'test'
}


# ------------------------------------------------------------
# 8. Match selected studies with their classification labels
# ------------------------------------------------------------

selected = manifest.merge(
    studies,
    on='StudyInstanceUID',
    how='left',
    validate='one_to_one'
)

# Every study in the manifest must exist in train.csv
assert selected.study_target.notna().all(), \
    'Manifest contains studies missing from train.csv.'

selected_ids = set(selected.StudyInstanceUID)


# ------------------------------------------------------------
# 9. Select all slices belonging to the selected studies
# ------------------------------------------------------------

data = labels[
    labels.StudyInstanceUID.isin(selected_ids)
].copy()


# ------------------------------------------------------------
# 10. Construct the full DICOM file paths
# ------------------------------------------------------------

data['dicom_path'] = [
    str(TRAIN_DIR / a / b / f'{c}.dcm')
    for a, b, c in data[keys].itertuples(
        index=False,
        name=None
    )
]


# ------------------------------------------------------------
# 11. Validate a reproducible sample of DICOM files
# ------------------------------------------------------------

probe = data.sample(
    min(16, len(data)),
    random_state=SEED
)

for row in probe.itertuples():

    path = Path(row.dicom_path)

    # Check that the DICOM file exists
    assert path.is_file(), f'Missing DICOM: {path}'

    # Read DICOM metadata without loading image pixels
    dcm = pydicom.dcmread(
        str(path),
        stop_before_pixels=True
    )

    # Check that DICOM metadata matches CSV identifiers
    assert str(dcm.StudyInstanceUID) == str(row.StudyInstanceUID)
    assert str(dcm.SeriesInstanceUID) == str(row.SeriesInstanceUID)
    assert str(dcm.SOPInstanceUID) == str(row.SOPInstanceUID)


# ------------------------------------------------------------
# 12. Original summary
# ------------------------------------------------------------

print({
    'all_csv_slices': len(labels),
    'all_studies': len(studies),
    'selected_studies': len(selected),
    'selected_slices_before_caps': len(data),
    'sample_DICOM_checks': len(probe)
})


# ============================================================
# ADDED: Detailed Dataset Summary (Informational Only)
# ============================================================

print("\n========== DATASET SPLIT SUMMARY ==========")

# Number of studies in each split
print("\nStudies per split:")
print(selected.groupby('split').size())

# Positive / negative study distribution
print("\nStudy class distribution:")
print(
    selected.groupby(['split', 'study_target'])
    .size()
    .unstack(fill_value=0)
)

# Number of selected image slices
print("\nTotal selected DICOM slices:", len(data))

# Confirm manifest location
print("\nManifest:", manifest_path)

print("===========================================")


c:\Users\PythonMamba\envs\AIProgram\Lib\site-packages\pydicom\valuerep.py:440: UserWarning: Invalid value for VR UI: '740b2e648ae0'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)
c:\Users\PythonMamba\envs\AIProgram\Lib\site-packages\pydicom\valuerep.py:440: UserWarning: Invalid value for VR UI: 'b0b8af8b65e6'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)
c:\Users\PythonMamba\envs\AIProgram\Lib\site-packages\pydicom\valuerep.py:440: UserWarning: Invalid value for VR UI: '2bd46a0d0f2f'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)
c:\Users\PythonMamba\envs\AIProgram\Lib\site-packages\pydicom\valuerep.py:440: UserWarning: Invalid value for VR UI: '4d33ed4cc6cd'. Please see <https://dicom.nema

{'all_csv_slices': 1790594, 'all_studies': 7279, 'selected_studies': 600, 'selected_slices_before_caps': 150904, 'sample_DICOM_checks': 16}

========== DATASET SPLIT SUMMARY ==========

Studies per split:
split
test           90
train         420
validation     90
dtype: int64

Study class distribution:
study_target    0    1
split                 
test           61   29
train         283  137
validation     61   29

Total selected DICOM slices: 150904

Manifest: C:\RSNA_Project\results\split_manifest.csv


c:\Users\PythonMamba\envs\AIProgram\Lib\site-packages\pydicom\valuerep.py:440: UserWarning: Invalid value for VR UI: 'e27d37090422'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)
c:\Users\PythonMamba\envs\AIProgram\Lib\site-packages\pydicom\valuerep.py:440: UserWarning: Invalid value for VR UI: '2be21e3377dc'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)
c:\Users\PythonMamba\envs\AIProgram\Lib\site-packages\pydicom\valuerep.py:440: UserWarning: Invalid value for VR UI: '8d24feaeaed9'. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)


## 3. Use the saved study-level train/validation/test split

Patient identifiers were absent from the inspected DICOM headers. The completed run therefore guarantees that a study appears in only one partition, but cannot verify patient-level separation across different studies. This is an explicit limitation, not a patient-grouped split claim.

In [13]:

# ============================================================
# STEP 3: Study-Level Split Audit
# ============================================================

# 1. Define the dataset splits
split_names = ['train', 'validation', 'test']

splits = {
    name: set(
        manifest.loc[
            manifest.split == name,
            'StudyInstanceUID'
        ]
    )
    for name in split_names
}


# 2. Verify there is no study overlap between splits
assert all(
    splits[a].isdisjoint(splits[b])
    for i, a in enumerate(split_names)
    for b in split_names[i+1:]
)


# 3. Verify all selected studies are included
assert set.union(*splits.values()) == selected_ids


# 4. Create study-level table
study_table = selected[
    ['StudyInstanceUID', 'study_target']
].copy()


# 5. Audit class distribution in every split
for name, ids in splits.items():

    sub = study_table[
        study_table.StudyInstanceUID.isin(ids)
    ]

    positives = int(sub.study_target.sum())
    negatives = int(len(sub) - positives)

    print({
        'split': name,
        'studies': len(sub),
        'positive_studies': positives,
        'negative_studies': negatives,
        'study_PE_rate': float(sub.study_target.mean())
    })


# 6. Save the split manifest as JSON
with open(OUTPUT_DIR / 'split_manifest.json', 'w') as f:
    json.dump(
        {k: sorted(v) for k, v in splits.items()},
        f,
        indent=2
    )


# 7. Original audit message
print(
    'Study-level split audit passed. '
    'Patient-level separation is unverified.'
)


# ============================================================
# ADDED: Additional verification summary
# Does not modify any original data or algorithm.
# ============================================================

print("\n========== SPLIT VERIFICATION ==========")

print("Train studies:", len(splits['train']))
print("Validation studies:", len(splits['validation']))
print("Test studies:", len(splits['test']))

print(
    "Train / Validation overlap:",
    len(splits['train'] & splits['validation'])
)

print(
    "Train / Test overlap:",
    len(splits['train'] & splits['test'])
)

print(
    "Validation / Test overlap:",
    len(splits['validation'] & splits['test'])
)

print("Manifest saved to:", OUTPUT_DIR / 'split_manifest.json')

print("========================================")


{'split': 'train', 'studies': 420, 'positive_studies': 137, 'negative_studies': 283, 'study_PE_rate': 0.3261904761904762}
{'split': 'validation', 'studies': 90, 'positive_studies': 29, 'negative_studies': 61, 'study_PE_rate': 0.32222222222222224}
{'split': 'test', 'studies': 90, 'positive_studies': 29, 'negative_studies': 61, 'study_PE_rate': 0.32222222222222224}
Study-level split audit passed. Patient-level separation is unverified.

========== SPLIT VERIFICATION ==========
Train studies: 420
Validation studies: 90
Test studies: 90
Train / Validation overlap: 0
Train / Test overlap: 0
Validation / Test overlap: 0
Manifest saved to: C:\RSNA_Project\results\split_manifest.json


## 4. Cap slices and cache windowed images

All positive slices in selected studies are retained. Negative slices are deterministically capped at 24 per training study and 12 per validation/test study, matching the completed prototype configuration. Thus evaluation is on sampled slices, not complete CT studies.

In [18]:
 
# ============================================================
# STEP 4: Slice Sampling and DICOM Image Preprocessing
# ============================================================

# ------------------------------------------------------------
# 1. Sample negative slices while retaining all positives
# ------------------------------------------------------------

def cap_rows(frame, negative_cap):

    chunks = []

    for _, g in frame.groupby('StudyInstanceUID', sort=False):

        pos = g[g.pe_present_on_image == 1]
        neg = g[g.pe_present_on_image == 0]

        if len(neg) > negative_cap:
            neg = neg.sample(
                n=negative_cap,
                random_state=SEED
            )

        chunks.append(pd.concat([pos, neg]))

    return pd.concat(chunks, ignore_index=True)


# ------------------------------------------------------------
# 2. Apply sampling independently to each split
# ------------------------------------------------------------

train_rows = cap_rows(
    data[data.StudyInstanceUID.isin(splits['train'])],
    MAX_NEG_TRAIN_PER_STUDY
)

val_rows = cap_rows(
    data[data.StudyInstanceUID.isin(splits['validation'])],
    MAX_NEG_EVAL_PER_STUDY
)

test_rows = cap_rows(
    data[data.StudyInstanceUID.isin(splits['test'])],
    MAX_NEG_EVAL_PER_STUDY
)


# ------------------------------------------------------------
# 3. Combine selected slices and remove duplicate SOP IDs
# ------------------------------------------------------------

selected_rows = pd.concat(
    [train_rows, val_rows, test_rows],
    ignore_index=True
).drop_duplicates(
    'SOPInstanceUID'
).reset_index(drop=True)


# ORIGINAL: Print sampling results
print({
    'train_slices': len(train_rows),
    'validation_slices_sampled': len(val_rows),
    'test_slices_sampled': len(test_rows),
    'unique_images_to_cache': len(selected_rows)
})


# ============================================================
# ADDED: Estimate cache storage requirements
# This does not change the sampling or preprocessing.
# ============================================================

bytes_per_image = IMAGE_SIZE * IMAGE_SIZE
estimated_bytes = len(selected_rows) * bytes_per_image

estimated_gb = estimated_bytes / (1024 ** 3)

# Includes only the uint8 pixel arrays.
# Actual .npy files also contain a small header.

disk_info = shutil.disk_usage(LOCAL_CACHE)
free_gb = disk_info.free / (1024 ** 3)

print("\n========== CACHE STORAGE ESTIMATE ==========")
print("Selected images:", len(selected_rows))
print("Image dimensions:", IMAGE_SIZE, "x", IMAGE_SIZE)
print("Estimated pixel data (GiB):", round(estimated_gb, 2))
print("Available disk space (GiB):", round(free_gb, 2))
print("Cache directory:", LOCAL_CACHE)
print("================================================")

# ADDED: Stop if the estimated cache is too large
# Keep 20 GiB free for other operations.
assert disk_info.free - estimated_bytes > 20 * (1024 ** 3), (
    "Insufficient free disk space for cache with 20 GiB reserve."
)

# ADDED: Prefer locally downloaded DICOM files.
# Original dataset paths and sampling remain unchanged.

LOCAL_DICOM_DIR = Path(r"D:\RSNA_DICOM")

def resolve_dicom_path(original_path):
    relative_path = Path(original_path).relative_to(TRAIN_DIR)
    local_path = LOCAL_DICOM_DIR / relative_path

    if local_path.is_file():
        return str(local_path)

    return str(original_path)

# ------------------------------------------------------------
# 4. Convert DICOM pixel data to windowed uint8 image
# ------------------------------------------------------------

def window_to_uint8(path):

    # MODIFIED: Prefer the local DICOM copy.
    dcm = pydicom.dcmread(resolve_dicom_path(path))

    pixels = dcm.pixel_array.astype(np.float32)

    # Convert stored pixel values to Hounsfield Units
    hu = (
        pixels * float(getattr(dcm, 'RescaleSlope', 1.0))
        + float(getattr(dcm, 'RescaleIntercept', 0.0))
    )

    # Apply CT window
    low = WINDOW_CENTER - WINDOW_WIDTH / 2
    high = WINDOW_CENTER + WINDOW_WIDTH / 2

    arr = np.clip(
        (hu - low) / (high - low),
        0,
        1
    )

    # Handle inverted grayscale DICOM images
    if str(
        getattr(dcm, 'PhotometricInterpretation', 'MONOCHROME2')
    ) == 'MONOCHROME1':
        arr = 1 - arr

    # Convert to uint8 and resize to CNN input dimensions
    im = Image.fromarray(
        np.uint8(np.round(arr * 255))
    ).resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        Image.Resampling.BILINEAR
    )

    return np.asarray(im, dtype=np.uint8)


# ------------------------------------------------------------
# 5. Decode and cache selected DICOM images
# ------------------------------------------------------------

cache_paths = []

for row in tqdm(
    selected_rows.itertuples(),
    total=len(selected_rows),
    desc='Decode/window selected DICOMs'
):

    dst = LOCAL_CACHE / f'{row.SOPInstanceUID}.npy'

    if not dst.is_file():
        np.save(
            dst,
            window_to_uint8(row.dicom_path),
            allow_pickle=False
        )

    cache_paths.append(str(dst))


# ------------------------------------------------------------
# 6. Attach cached image paths to the selected data
# ------------------------------------------------------------

selected_rows['cache_path'] = cache_paths

path_by_sop = (
    selected_rows.set_index('SOPInstanceUID')
    .cache_path.to_dict()
)

for frame in (train_rows, val_rows, test_rows):
    frame['cache_path'] = frame.SOPInstanceUID.map(path_by_sop)


# ------------------------------------------------------------
# 7. Verify cache paths are assigned to every selected slice
# ------------------------------------------------------------

assert (
    train_rows.cache_path.notna().all()
    and val_rows.cache_path.notna().all()
    and test_rows.cache_path.notna().all()
)


# ORIGINAL: Cache completion message
print(
    'Local cache ready:',
    LOCAL_CACHE,
    'files:',
    len(list(LOCAL_CACHE.glob('*.npy')))
)


{'train_slices': 15630, 'validation_slices_sampled': 2249, 'test_slices_sampled': 2366, 'unique_images_to_cache': 20245}

========== CACHE STORAGE ESTIMATE ==========
Selected images: 20245
Image dimensions: 224 x 224
Estimated pixel data (GiB): 0.95
Available disk space (GiB): 161.87
Cache directory: D:\RSNA_Cache


Decode/window selected DICOMs:   0%|          | 0/20245 [00:00<?, ?it/s]

Local cache ready: D:\RSNA_Cache files: 20245


In [15]:
# ADDED: Export the original selected DICOM file list.
# Does not modify sampling, labels, or data splits.

from pathlib import Path

download_list = Path(
    r"C:\RSNA_Project\dicom_download_list.txt"
)

relative_paths = [
    Path(p).relative_to(TRAIN_DIR).as_posix()
    for p in selected_rows["dicom_path"]
]

download_list.write_text(
    "\n".join(relative_paths) + "\n",
    encoding="utf-8"
)

print("Files selected:", len(relative_paths))
print("Download list:", download_list)
print("First 3 files:", relative_paths[:3])

Files selected: 20245
Download list: C:\RSNA_Project\dicom_download_list.txt
First 3 files: ['6cc79b883538/57e3520a9db2/5bd03ffe234d.dcm', '6cc79b883538/57e3520a9db2/546f38fe2687.dcm', '6cc79b883538/57e3520a9db2/e633a437e2fe.dcm']


## 5. Pretrained ResNet18 and efficient loaders

The GPU is used automatically when available. Mixed precision is enabled on CUDA. If pretrained weights cannot be downloaded, the notebook stops rather than silently training from random weights.


In [19]:
MEAN=torch.tensor([.485,.456,.406],dtype=torch.float32)[:,None,None]
STD=torch.tensor([.229,.224,.225],dtype=torch.float32)[:,None,None]
class SliceDataset(Dataset):
    def __init__(self,frame): self.frame=frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        r=self.frame.iloc[i]
        gray=np.load(r.cache_path,allow_pickle=False).astype(np.float32)/255.0
        x=torch.from_numpy(gray).unsqueeze(0).expand(3,-1,-1).contiguous()
        x=(x-MEAN)/STD
        return x,torch.tensor(float(r.pe_present_on_image)),str(r.StudyInstanceUID),str(r.SOPInstanceUID)

def loader(frame,shuffle=False):
    return DataLoader(SliceDataset(frame),batch_size=BATCH_SIZE,shuffle=shuffle,num_workers=NUM_WORKERS,
                      pin_memory=DEVICE.type=='cuda',persistent_workers=NUM_WORKERS>0)

def new_model(pretrained=True):
    weights=models.ResNet18_Weights.DEFAULT if pretrained else None
    model=models.resnet18(weights=weights)
    model.fc=nn.Linear(model.fc.in_features,1)
    return model

try:
    probe_model=new_model(pretrained=True).to(DEVICE)
    test_x=SliceDataset(train_rows)[0][0].unsqueeze(0).to(DEVICE)
    with torch.inference_mode(): probe_logits=probe_model(test_x)
    assert tuple(probe_logits.shape)==(1,1)
    initial_state={k:v.detach().cpu().clone() for k,v in probe_model.state_dict().items()}
    del probe_model, test_x, probe_logits
    if torch.cuda.is_available(): torch.cuda.empty_cache()
except Exception as e:
    raise RuntimeError('Could not load pretrained ResNet18 or run its forward pass. Check Colab Internet, then rerun.') from e
print('ResNet18 pretrained forward pass passed on',DEVICE)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\PythonMamba\User/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 164MB/s]


ResNet18 pretrained forward pass passed on cuda


## 6. Compare frozen features with partial fine-tuning

Both runs start from identical pretrained weights and use the same patient split. The validation slice ROC AUC selects the epoch and model. The test set is evaluated only after selection.


In [ ]:

# ============================================================
# STEP 6: Train and Compare ResNet18 Configurations
# ============================================================

# ------------------------------------------------------------
# 1. Safely calculate AUC / Average Precision
# ------------------------------------------------------------

def auc_or_none(y, p, fn):
    return float(fn(y, p)) if len(np.unique(y)) > 1 else None


# ------------------------------------------------------------
# 2. Calculate classification metrics
# ------------------------------------------------------------

def metrics(frame):

    y = frame.y_true.to_numpy(dtype=int)
    p = frame.probability.to_numpy(dtype=float)
    pred = (p >= .5).astype(int)

    return {
        'n': int(len(frame)),
        'positive_rate': float(y.mean()),
        'roc_auc': auc_or_none(y, p, roc_auc_score),
        'average_precision': auc_or_none(y, p, average_precision_score),
        'accuracy_at_0_5': float(accuracy_score(y, pred)),
        'precision_at_0_5': float(precision_score(y, pred, zero_division=0)),
        'recall_at_0_5': float(recall_score(y, pred, zero_division=0)),
        'f1_at_0_5': float(f1_score(y, pred, zero_division=0)),
        'confusion_matrix_at_0_5': confusion_matrix(
            y, pred, labels=[0, 1]
        ).tolist()
    }


# ------------------------------------------------------------
# 3. Calculate weighted validation BCE loss
# ------------------------------------------------------------

def validation_weighted_bce(model, dl, loss_fn):

    model.eval()
    losses = []

    with torch.inference_mode():

        for x, y, _, _ in dl:

            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            with torch.autocast(
                device_type='cuda',
                dtype=torch.float16,
                enabled=DEVICE.type == 'cuda'
            ):
                logits = model(x).squeeze(1)

            losses.append(
                float(loss_fn(logits.float(), y).cpu())
            )

    return float(np.mean(losses))


# ------------------------------------------------------------
# 4. Generate model predictions
# ------------------------------------------------------------

def predict(model, dl, desc):

    model.eval()

    ys = []
    ps = []
    studies_out = []
    sops = []

    with torch.inference_mode():

        for x, y, st, sp in tqdm(dl, desc=desc, leave=False):

            x = x.to(DEVICE, non_blocking=True)

            with torch.autocast(
                device_type='cuda',
                dtype=torch.float16,
                enabled=DEVICE.type == 'cuda'
            ):
                logits = model(x).squeeze(1)

            ps.extend(
                torch.sigmoid(logits.float()).cpu().numpy().tolist()
            )

            ys.extend(y.numpy().astype(int).tolist())
            studies_out.extend(st)
            sops.extend(sp)

    return pd.DataFrame({
        'StudyInstanceUID': studies_out,
        'SOPInstanceUID': sops,
        'y_true': ys,
        'probability': ps
    })


# ------------------------------------------------------------
# 5. Create training and validation DataLoaders
# ------------------------------------------------------------

train_dl = loader(train_rows, True)
val_dl = loader(val_rows, False)


# ------------------------------------------------------------
# 6. Train one ResNet18 configuration
# ------------------------------------------------------------

def fit(mode):

    torch.manual_seed(SEED)

    # Load ResNet18 with the same initial pretrained weights
    model = new_model(pretrained=False).to(DEVICE)
    model.load_state_dict(initial_state)

    # Freeze all model parameters
    for p in model.parameters():
        p.requires_grad = False

    # Train the final classification layer
    for p in model.fc.parameters():
        p.requires_grad = True

    # Additionally train layer4 for partial fine-tuning
    if mode == 'partial_finetune':
        for p in model.layer4.parameters():
            p.requires_grad = True

    # Calculate class imbalance
    y = train_rows.pe_present_on_image.to_numpy(dtype=int)

    positives = int(y.sum())
    negatives = len(y) - positives

    assert positives and negatives, (
        'Training sample must contain positive and negative slices.'
    )

    # Weighted Binary Cross Entropy
    loss_fn = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(
            [min(50., negatives / positives)],
            device=DEVICE
        )
    )

    # Different learning rates for the two strategies
    lr = 1e-3 if mode == 'frozen' else 1e-4

    # AdamW optimizer
    opt = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr,
        weight_decay=1e-4
    )

    # Automatic Mixed Precision
    scaler = torch.amp.GradScaler(
        'cuda',
        enabled=DEVICE.type == 'cuda'
    )

    best = -1.
    history = []

    ckpt = OUTPUT_DIR / 'checkpoints' / f'{mode}_best.pt'

    # --------------------------------------------------------
    # 7. Epoch training loop
    # --------------------------------------------------------

    for epoch in range(1, EPOCHS + 1):

        model.train()
        losses = []

        for x, yb, _, _ in tqdm(
            train_dl,
            desc=f'{mode} epoch {epoch}/{EPOCHS}'
        ):

            x = x.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type='cuda',
                dtype=torch.float16,
                enabled=DEVICE.type == 'cuda'
            ):
                logits = model(x).squeeze(1)
                loss = loss_fn(logits, yb)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            losses.append(float(loss.detach().cpu()))

        # ----------------------------------------------------
        # 8. Validate after each epoch
        # ----------------------------------------------------

        val_loss = validation_weighted_bce(
            model, val_dl, loss_fn
        )

        val_pred = predict(
            model,
            val_dl,
            f'{mode} validation'
        )

        vm = metrics(val_pred)

        score = vm['roc_auc']
        selection = -1. if score is None else score

        history.append({
            'epoch': epoch,
            'train_loss': float(np.mean(losses)),
            'validation_loss_weighted_bce': val_loss,
            'validation': vm
        })

        print(
            mode,
            'epoch', epoch,
            'mean loss', round(float(np.mean(losses)), 4),
            'validation ROC AUC', score
        )

        # ----------------------------------------------------
        # 9. Save best checkpoint
        # ----------------------------------------------------

        if selection > best:

            best = selection

            torch.save({
                'mode': mode,
                'epoch': epoch,
                'state_dict': model.state_dict(),
                'validation_metrics': vm
            }, ckpt)

    # --------------------------------------------------------
    # 10. Release GPU memory
    # --------------------------------------------------------

    del model, opt, scaler

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        'mode': mode,
        'checkpoint': str(ckpt),
        'best_validation_slice_roc_auc': None if best < 0 else best,
        'history': history
    }


# ============================================================
# 11. Run both ResNet18 experiments
# ============================================================

runs = [
    fit('frozen'),
    fit('partial_finetune')
]


# ------------------------------------------------------------
# 12. Save experiment results
# ------------------------------------------------------------

with open(OUTPUT_DIR / 'training_runs.json', 'w') as f:
    json.dump(
        runs,
        f,
        indent=2,
        allow_nan=False
    )


## 7. Select on validation and report held-out subset results

Study probability is the maximum predicted slice probability. The fixed 0.5 threshold is a prototype convention, not a clinical operating point. Outputs and checkpoints are written to `RSNA_PE_results` in Drive.


In [ ]:

# ============================================================
# STEP 7: Final Model Selection and Test Evaluation
# ============================================================

# ------------------------------------------------------------
# 1. Select best model using Validation Slice ROC AUC
# ------------------------------------------------------------

chosen = max(
    runs,
    key=lambda r: (
        -1.
        if r['best_validation_slice_roc_auc'] is None
        else r['best_validation_slice_roc_auc']
    )
)

# ADDED: Display the selected experiment
print("Selected model:", chosen['mode'])
print(
    "Best Validation Slice ROC AUC:",
    chosen['best_validation_slice_roc_auc']
)


# ------------------------------------------------------------
# 2. Load the best saved checkpoint
# ------------------------------------------------------------

saved = torch.load(
    chosen['checkpoint'],
    map_location=DEVICE,
    weights_only=False
)

final_model = new_model(pretrained=False).to(DEVICE)

final_model.load_state_dict(saved['state_dict'])


# ------------------------------------------------------------
# 3. Generate predictions on the held-out Test dataset
# ------------------------------------------------------------

test_pred = predict(
    final_model,
    loader(test_rows, False),
    'Held-out test subset'
)


# ------------------------------------------------------------
# 4. Retrieve the true study-level labels
# ------------------------------------------------------------

truth = (
    data[data.StudyInstanceUID.isin(splits['test'])]
    .groupby('StudyInstanceUID')
    .study_target.first()
)


# ------------------------------------------------------------
# 5. Aggregate slice predictions to study-level predictions
# ------------------------------------------------------------

study_pred = (
    test_pred.groupby('StudyInstanceUID')
    .probability.max()
    .rename('probability')
    .to_frame()
    .join(truth.rename('y_true'))
    .reset_index()
)


# ============================================================
# ADDED: Validate test prediction completeness
# Does not modify any predictions or scoring.
# ============================================================

assert len(test_pred) == len(test_rows), (
    "Test slice prediction count does not match test dataset."
)

assert study_pred.StudyInstanceUID.nunique() == len(splits['test']), (
    "Test study prediction count does not match saved split."
)

assert study_pred.y_true.notna().all(), (
    "Missing study-level ground truth labels."
)

print("Test slices evaluated:", len(test_pred))
print("Test studies evaluated:", len(study_pred))


# ------------------------------------------------------------
# 6. Calculate final evaluation metrics
# ------------------------------------------------------------

result = {
    'status': 'measured',
    'selected_mode': chosen['mode'],
    'selection_metric': 'validation slice ROC AUC',
    'best_validation_slice_roc_auc':
        chosen['best_validation_slice_roc_auc'],
    'test_slice_metrics': metrics(test_pred),
    'test_study_metrics': metrics(study_pred),
    'study_aggregation': 'maximum slice probability',
    'test_studies': int(study_pred.StudyInstanceUID.nunique()),
    'selected_studies': int(len(selected_ids)),
    'limits': (
        'sampled studies and sampled negative slices; '
        'prototype metrics only'
    )
}


# ------------------------------------------------------------
# 7. Save test predictions to CSV
# ------------------------------------------------------------

test_pred.to_csv(
    OUTPUT_DIR / 'test_slice_predictions.csv',
    index=False
)

study_pred.to_csv(
    OUTPUT_DIR / 'test_study_predictions.csv',
    index=False
)


# ------------------------------------------------------------
# 8. Save final results to JSON
# ------------------------------------------------------------

with open(OUTPUT_DIR / 'results.json', 'w') as f:

    json.dump(
        result,
        f,
        indent=2,
        allow_nan=False
    )


# ------------------------------------------------------------
# 9. Original results output
# ------------------------------------------------------------

print(json.dumps(
    result,
    indent=2,
    allow_nan=False
))

print('Saved outputs to', OUTPUT_DIR)


# ============================================================
# ADDED: Final evaluation summary
# ============================================================

print("\n========== FINAL TEST SUMMARY ==========")

print("Selected model:", result['selected_mode'])

print(
    "Validation Slice ROC AUC:",
    result['best_validation_slice_roc_auc']
)

print(
    "Test Slice ROC AUC:",
    result['test_slice_metrics']['roc_auc']
)

print(
    "Test Study ROC AUC:",
    result['test_study_metrics']['roc_auc']
)

print(
    "Study Confusion Matrix:",
    result['test_study_metrics']['confusion_matrix_at_0_5']
)

print("========================================")


## Review notes and reproducibility

The historical run did not save validation BCE loss, so this notebook does not invent a historical validation-loss curve. The training code above records weighted validation BCE on a future rerun. See the companion README and figures for measured historical results, ROC curves, confusion matrices, and the recorded training-loss curves. The test metrics are from sampled studies/slices and are prototype results only.

## Load a saved checkpoint for inference

After running setup and defining `new_model`, load the selected checkpoint (or replace the path with `frozen_best.pt`). Checkpoint files are in the Drive review folder.


In [4]:


# ============================================================
# STEP 8: Load a Saved ResNet18 Checkpoint for Inference
# Environment: Windows Server 2025 + NVIDIA Tesla T4
# ============================================================

# ------------------------------------------------------------
# 1. Define the checkpoint path
# ------------------------------------------------------------

CHECKPOINT = (
    OUTPUT_DIR / "checkpoints" / "partial_finetune_best.pt"
)


# ============================================================
# ADDED: Verify that the checkpoint exists
# ============================================================

assert CHECKPOINT.is_file(), (
    f"Checkpoint file not found: {CHECKPOINT}"
)

print("Checkpoint found:", CHECKPOINT)


# ------------------------------------------------------------
# 2. Load the saved checkpoint
# ------------------------------------------------------------

# Load only checkpoints from trusted sources.
saved = torch.load(
    CHECKPOINT,
    map_location=DEVICE,
    weights_only=False
)


# ------------------------------------------------------------
# 3. Recreate the ResNet18 architecture
# ------------------------------------------------------------

model = new_model(
    pretrained=False
).to(DEVICE)


# ------------------------------------------------------------
# 4. Load the trained model parameters
# ------------------------------------------------------------

model.load_state_dict(
    saved["state_dict"]
)


# ------------------------------------------------------------
# 5. Switch the model to evaluation mode
# ------------------------------------------------------------

model.eval()


# ------------------------------------------------------------
# 6. Original confirmation message
# ------------------------------------------------------------

print(
    "Loaded",
    saved["mode"],
    "epoch",
    saved["epoch"]
)


# ============================================================
# ADDED: Display checkpoint information
# ============================================================

print("\n========== LOADED MODEL ==========")
print("Architecture: ResNet18")
print("Training mode:", saved["mode"])
print("Saved epoch:", saved["epoch"])
print("Device:", DEVICE)
print("Checkpoint:", CHECKPOINT)
print("==================================")


NameError: name 'OUTPUT_DIR' is not defined